# 01 — Test frontend (état container + bundle servable)

Smoke test du container `fxvol-frontend` (10/10, dernier de la série smoke) — étape 1/5. Valide les **fondations** avant les tests Playwright qui demandent un DOM rendu : container UP, healthcheck OK, image attendue, port 8080 interne répond, bundle React accessible avec `<div id="root">` et un `<script type="module">`.

**Spécificités** :
- Image base = `nginx:alpine` (pas du Python custom). HC = `wget http://127.0.0.1:8080/`.
- Pas d'env vars métier à valider (le container ne fait que servir des fichiers statiques) — pas de §4 "env vars" comme pour les engines.
- Le bundle React est dans `/usr/share/nginx/html/` à l'intérieur du container ; le HTML root référence un script hashé `assets/index-XXXX.js` (cache busting).

**Couvre** :
1. Container présent et `running`
2. Docker healthcheck = `healthy`
3. Image attendue (`fx-options-frontend:local` ou override) + StartedAt + restart count
4. Port 8080 interne répond 200 et sert un HTML React (présence `<div id="root">`)
5. Le HTML référence un script `assets/index-<hash>.js` (cache busting actif)
6. Logs nginx propres (pas de `error` / `5xx` dans les 50 dernières lignes)

**Préreq** :
- Stack démarrée (le frontend n'a pas de profile, il monte par défaut) : `docker compose up -d`
- Pas besoin de l'api ou des engines pour ce notebook (on teste le serveur statique en isolation)

**Référence** : `docker-compose.yml § frontend`, `infrastructure/docker/Dockerfile.web`, `infrastructure/nginx/frontend.conf`.

## Setup

Note : le container Docker s'appelle `fxvol-frontend`, le service compose aussi `frontend`. Cohérent avec `fxvol-api` / `fxvol-nginx` côté serveur HTTP.

In [1]:
import re
import subprocess

CONTAINER = "fxvol-frontend"
results = []

def record(label, ok, detail=""):
    results.append((label, ok, detail))
    sym = "OK" if ok else "FAIL"
    print(f"  [{sym:4}] {label}{('  | ' + detail) if detail else ''}")

def docker_inspect(fmt):
    out = subprocess.run(
        ["docker", "inspect", "-f", fmt, CONTAINER],
        capture_output=True, text=True,
    )
    return out.stdout.strip()

def docker_exec(*cmd):
    out = subprocess.run(
        ["docker", "exec", CONTAINER, *cmd],
        capture_output=True, text=True,
    )
    return out.returncode, out.stdout, out.stderr

print(f"target container = {CONTAINER}")

target container = fxvol-frontend


## 1. Container présent et `running`

**Ce que tu dois voir** : state = `running`. Si `<not found>` → stack pas démarrée. Si `restarting` → crashloop nginx (typiquement bundle absent ou conf nginx invalide).

In [2]:
print("== 1. container state ==")

state = docker_inspect("{{.State.Status}}")
record("docker container state", state == "running", state or "<not found>")

restart_count = docker_inspect("{{.RestartCount}}")
record("restart count raisonnable (≤ 3)",
       restart_count.isdigit() and int(restart_count) <= 3,
       f"restart count = {restart_count}")

== 1. container state ==
  [OK  ] docker container state  | running
  [OK  ] restart count raisonnable (≤ 3)  | restart count = 0


## 2. Docker healthcheck = `healthy`

Le healthcheck (`docker-compose.yml § frontend`) fait `wget -qO- http://127.0.0.1:8080/` toutes les 15s, timeout 3s, 3 retries, start_period 5s. Très court parce que nginx démarre instantanément (pas d'init Python lourd).

**Healthy ⇒ bundle servable** : le HC tape la racine, qui répond avec `index.html` si le bundle est bien dans `/usr/share/nginx/html/`. Si bundle absent (build cassé), `wget` 404 → exit ≠ 0 → unhealthy.

In [3]:
print("== 2. healthcheck ==")

health = docker_inspect("{{.State.Health.Status}}")
record("docker healthcheck", health == "healthy", health or "<no healthcheck>")

last_log = docker_inspect("{{(index .State.Health.Log 0).Output}}")
if last_log:
    snippet = last_log[:120].replace("\n", " ")
    print(f"  [INFO] dernière sortie HC : {snippet!r}{'...' if len(last_log) > 120 else ''}")

== 2. healthcheck ==
  [OK  ] docker healthcheck  | healthy
  [INFO] dernière sortie HC : '<!doctype html> <html lang="en">   <head>     <meta charset="UTF-8" />     <meta name="viewport" content="width=device-w'...


## 3. Image + StartedAt (info)

Pas un test pass/fail — juste de l'info pour le diag. L'image attendue est `fx-options-frontend:local` (build local) ou un tag override si registry remote.

In [4]:
print("== 3. image + uptime info ==")

image = docker_inspect("{{.Config.Image}}")
started_at = docker_inspect("{{.State.StartedAt}}")
print(f"  [INFO] image      : {image}")
print(f"  [INFO] StartedAt  : {started_at}")

expected_prefix = "fx-options-frontend"
record(f"image préfixe '{expected_prefix}'",
       expected_prefix in image,
       image)

== 3. image + uptime info ==
  [INFO] image      : fx-options-frontend:local
  [INFO] StartedAt  : 2026-04-29T09:01:52.881918723Z
  [OK  ] image préfixe 'fx-options-frontend'  | fx-options-frontend:local


## 4. Port 8080 interne répond + sert le HTML React

On tape `http://127.0.0.1:8080/` **depuis l'intérieur du container** (via `docker exec wget`) — c'est ce que fait le HC mais on inspecte le contenu, pas juste le code de retour.

**Ce qu'on doit voir** : un HTML qui contient `<!DOCTYPE html>` + `<div id="root">` (le mount point React). Si le bundle est absent ou tronqué, le HTML peut être présent (page nginx default) mais sans le `<div id="root">` → la SPA ne montera jamais.

**Note** : le test depuis l'host (`http://localhost/` via le reverse nginx) est dans le notebook 02 — ici on isole pour distinguer un soucis frontend d'un soucis du proxy.

In [5]:
print("== 4. port 8080 interne sert le HTML React ==")

rc, body, err = docker_exec("wget", "-qO-", "http://127.0.0.1:8080/")
record("GET / sur port 8080 interne (exit 0)",
       rc == 0,
       f"exit={rc}" + (f", stderr={err.strip()[:100]!r}" if err else ""))

if rc == 0:
    has_doctype = "<!doctype html>" in body.lower() or "<!DOCTYPE html>" in body
    has_root_div = re.search(r'<div\s+id=["\']root["\']', body) is not None
    record("HTML contient <!DOCTYPE html>", has_doctype, f"size = {len(body)} bytes")
    record("HTML contient <div id=\"root\">", has_root_div,
           "OK" if has_root_div else f"head: {body[:200]!r}")

== 4. port 8080 interne sert le HTML React ==
  [OK  ] GET / sur port 8080 interne (exit 0)  | exit=0
  [OK  ] HTML contient <!DOCTYPE html>  | size = 402 bytes
  [OK  ] HTML contient <div id="root">  | OK


## 5. Cache busting actif — script hashé

Vite emet le bundle avec un hash dans le nom (`assets/index-<8chars>.js`) pour le cache busting : à chaque build qui change le code, le hash change → le browser invalide automatiquement le cache. Si le hash manque ou est constant, un user qui revient après un deploy peut voir l'ancienne version pendant des heures.

Pattern attendu : `<script type="module" crossorigin src="/assets/index-XXXXXXXX.js">` avec 8+ chars hex.

In [6]:
print("== 5. cache busting (hash dans le filename) ==")

if rc != 0 or not body:
    record("hash dans script tag", False, "skip (cf. §4)")
else:
    # Vite : <script type="module" crossorigin src="/assets/index-XXXXXXXX.js"></script>
    # On accepte 6+ chars dans le hash pour être tolérant.
    m = re.search(r'<script[^>]+src="(/assets/index-[a-zA-Z0-9_-]{6,}\.js)"', body)
    record("<script src=\"/assets/index-<hash>.js\"> trouvé",
           m is not None,
           f"src = {m.group(1)!r}" if m else "introuvable — bundle pas hashé ou pas servi")

== 5. cache busting (hash dans le filename) ==
  [OK  ] <script src="/assets/index-<hash>.js"> trouvé  | src = '/assets/index-Bc4oxzX_.js'


## 6. Logs nginx propres

On regarde les 50 dernières lignes : aucun `error` / `failed` / `5xx`. Les logs nginx sont écrits sur stdout/stderr du container (config standard `nginx:alpine`).

**Ce qui doit alerter** :
- `connect() failed` → upstream API joignable indique un soucis avec `nginx-public`, mais ce n'est pas le scope frontend
- `404 Not Found` répétés sur `/assets/...` → bundle build cassé
- `error` au boot → mauvaise config nginx (vérifier `infrastructure/nginx/frontend.conf`)

In [7]:
print("== 6. logs nginx 50 dernières lignes ==")

logs = subprocess.run(
    ["docker", "logs", "--tail", "50", CONTAINER],
    capture_output=True, text=True,
)
log_text = (logs.stdout + logs.stderr).lower()

# Patterns critiques (ignore les 200 / GET routine).
bad_patterns = ["[error]", "emerg", " 500 ", " 502 ", " 503 ", " 504 "]
found = [p for p in bad_patterns if p in log_text]
record("aucun pattern d'erreur dans les 50 dernières lignes",
       not found,
       f"found = {found}" if found else "clean")

if found:
    print("  [INFO] sample logs (dernières 20 lignes pour debug) :")
    for line in (logs.stdout + logs.stderr).splitlines()[-20:]:
        print(f"    {line}")

== 6. logs nginx 50 dernières lignes ==
  [OK  ] aucun pattern d'erreur dans les 50 dernières lignes  | clean


## Récap final

In [8]:
n_ok = sum(1 for _, ok, _ in results if ok)
n_fail = sum(1 for _, ok, _ in results if not ok)

print(f"\n{'LABEL':<60} STATUS  DETAIL")
print("-" * 110)
for label, ok, detail in results:
    sym = "OK" if ok else "FAIL"
    print(f"{label:<60} {sym:<6}  {detail}")
print("-" * 110)
print(f"\n{n_ok} OK / {n_fail} FAIL  ({len(results)} total)")

if n_fail == 0:
    print("\nOK — container frontend sain. Pass au notebook 02 (assets statiques depuis l'host via reverse nginx).")


LABEL                                                        STATUS  DETAIL
--------------------------------------------------------------------------------------------------------------
docker container state                                       OK      running
restart count raisonnable (≤ 3)                              OK      restart count = 0
docker healthcheck                                           OK      healthy
image préfixe 'fx-options-frontend'                          OK      fx-options-frontend:local
GET / sur port 8080 interne (exit 0)                         OK      exit=0
HTML contient <!DOCTYPE html>                                OK      size = 402 bytes
HTML contient <div id="root">                                OK      OK
<script src="/assets/index-<hash>.js"> trouvé                OK      src = '/assets/index-Bc4oxzX_.js'
aucun pattern d'erreur dans les 50 dernières lignes          OK      clean
----------------------------------------------------------------

## Troubleshooting cheat sheet

| Symptôme | Cause probable | Fix |
|---|---|---|
| `state` = `<not found>` | Stack pas démarrée OU service `frontend` non listé dans le compose | `docker compose up -d frontend` |
| `state` = `restarting` ou restart count > 3 | Crashloop nginx : conf invalide ou bundle absent | `docker logs fxvol-frontend --tail 100` ; suspects : `infrastructure/nginx/frontend.conf` cassé, `dist/` vide dans le build |
| `healthcheck` = `unhealthy` | nginx tourne mais ne sert rien sur 8080 | `docker exec fxvol-frontend ls -la /usr/share/nginx/html` (doit contenir `index.html` + `assets/`) |
| HTML servi mais sans `<div id="root">` | `index.html` est la page nginx default → bundle pas copié dans la stage finale du Dockerfile | revoir `infrastructure/docker/Dockerfile.web` (instruction `COPY --from=build /app/dist /usr/share/nginx/html`) |
| script hash introuvable | build Vite KO (pas de cache busting) ou `index.html` ≠ celui du build | `docker compose build --no-cache frontend` puis `docker compose up -d frontend` |
| logs `error` au boot | conf nginx invalide | `docker exec fxvol-frontend nginx -t` pour valider la conf |
| `image` ≠ `fx-options-frontend:*` | Image custom (release pinnée) | OK si voulu, sinon revoir `${FRONTEND_IMAGE}` env var |